# Week 12: Visualizations

Let's import some libraries and helpers.

In [ ]:
!wget -qO- https://github.com/PSAM-5005-2026S-A/5005-utils/releases/latest/download/flowers.tar.gz | tar xz

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import requests

from os import listdir
from PIL import Image as PImage
from time import sleep

from keys import GEMINI_KEY, NYC_KEY

## Sort Flowers

We have $600$ images like this that we want to sort by color:

<img src="https://raw.githubusercontent.com/PSAM-5005-2026S-A/5005-utils/refs/heads/main/datasets/image/flowers/00_001.png" width="100px" />
<img src="https://raw.githubusercontent.com/PSAM-5005-2026S-A/5005-utils/refs/heads/main/datasets/image/flowers/01_001.png" width="100px" />
<img src="https://raw.githubusercontent.com/PSAM-5005-2026S-A/5005-utils/refs/heads/main/datasets/image/flowers/02_001.png" width="100px" />

Let's use an LLM to get a color category for each of our images.

Since this is a proof-of-concept, let's start by testing this method on $32$ files first.

In [ ]:
all_files = sorted(f for f in listdir("data/image/flowers") if f.endswith("png"))

step = int(len(all_files) // 32)

to_process = all_files[::step]

imgs = []

for f in to_process:
  imgs.append(PImage.open(f"data/image/flowers/{f}").convert("RGB").resize((64,64)))

### LLM

Let's ask Gemini to categorize our flowers based on color.

We'll give it a few options and ask for it to pick one of the choices.

In [ ]:
!pip install -U google-genai

In [ ]:
from google import genai
from pydantic import BaseModel
from typing import List

In [ ]:
client = genai.Client(api_key=GEMINI_KEY)

color_options = [
  "RED",
  "ORANGE",
  "YELLOW",
  "GREEN",
  "BLUE",
  "PURPLE",
  "PINK",
  "WHITE",
]

class Colors(BaseModel):
  colors: List[str]

### Image Prompts

In [ ]:
results = []

In [ ]:
for idx in range(0, len(imgs), 8):
  imgs8 = imgs[idx : idx + 8]
  fnames8 = to_process[idx : idx + 8]

  response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=[
      imgs8,
      color_options,
      "What are the colors of these flowers ? Answer as a list using using a single colors from the options given per image.",
    ],
    config={
      "response_mime_type": "application/json",
      "response_json_schema": Colors.model_json_schema(),
    },
  )

  sleep(1)

  response_data = response.to_json_dict()
  colors = json.loads(response_data["candidates"][0]["content"]["parts"][0]["text"])["colors"]

  for i,c in enumerate(colors):
    results.append([fnames8[i], c])


### Sort and Export json

In [ ]:
def by_color(A):
  return color_options.index(A[1])

sorted_results = sorted(results, key=by_color)

sorted_filenames = []
for d in sorted_results:
  sorted_filenames.append(d[0])

ofp = open("www/flowers.json", "w")
json.dump(sorted_filenames, ofp)
ofp.close()

## Filter 311

In [ ]:
URL = "https://data.cityofnewyork.us/api/v3/views/erm2-nwe9/query.json"
PAGE = "pageNumber=1"
ROWS = "pageSize=4096"
OPTIONS = "includeSystem=false"

soda_res = requests.get(f"{URL}?{PAGE}&{ROWS}&{OPTIONS}&app_token={NYC_KEY}")
soda_data = soda_res.json()

len(soda_data)

In [ ]:
to_drop = [
  "taxi_pick_up_location", "bridge_highway_name", "taxi_company_borough",
  "bridge_highway_segment", "bridge_highway_direction", "road_ramp",
  "landmark", "facility_type", ":@computed_region_92fq_4b7q", "street_name", "address_type",
  "cross_street_1", "cross_street_2", "intersection_street_1", "intersection_street_2",
  "location", ":@computed_region_f5dn_yrer", ":@computed_region_yeji_bk3q", ":@computed_region_sbqj_enih",
  "park_facility_name", "x_coordinate_state_plane", "y_coordinate_state_plane", "open_data_channel_type",
  "park_borough", "bbl", "council_district", "community_board", "police_precinct",
  "resolution_action_updated_date", "closed_date", "vehicle_type", "due_date",
  ":id", ":version", ":created_at", ":updated_at",
]

data_df = pd.DataFrame(soda_data).drop(columns=to_drop).astype({"latitude": float, "longitude": float})

In [ ]:
data_df[data_df["descriptor"] == "Graffiti"].to_json("www/graffiti.json", orient="records")